#Implimenting the EM Algorithm

## Setting Parameters

This notebook simulates a data set and then uses the EM Algorithm to estimate variance components and solve the mixed model equations.

In [ ]:
# The genetic variance used to simulate the data set
varA = 10
# The residual variance used to simulate the data
varE = 50
# The number of lines/varieties/hybrids simulated
nLines = 500
# The number of locations simulated
nLocs = 5
# Convergence criteria - if the change in alpha (varE/VarA) is less than this number then convergence is achieved - generally this is applied to the log likelihood and not the parameters
criteria = .00000001
# The starting value for alpha
alpha = 1

## Simulating the Data
Here we simulate a simple data set to demonstrate how the EM Algorithm works to simultaneously solve for the variance components and model effects.

In [ ]:
#Set up Z matrix
 Z=matrix(0,nLines*nLocs,nLines)
  count=1
  for(i in c(1:nLocs)){
    for(j in c(1:nLines)){
      Z[count,j]=1
      count=count+1
    }
  }
  #set up X
  X=matrix(0,nLines*nLocs,nLocs)
  count=1
  for(i in c(1:nLocs)){
    for(j in c(1:nLines)){
      X[count,i]=1
      count=count+1
    }
  }

  #sample residual
  res=rnorm(nLines*nLocs,0,varE**.5)
  #sample line effects
  lines=rnorm(nLines,0,varA**.5)
  #sample loc effects
  locs=rnorm(nLocs,150,40)
  #contruct phenotypes
  Y=X%*%locs+Z%*%lines+res
  trueVarA = var(lines)
  trueVarE = var(res)

## Simulated Values

Since the realized values of the variance components will not be identical to the simulation parameters let's show the realized values to compare to the values estimated via the EM Algorithm.


Realized Genetic Variance:


In [ ]:
trueVarA

Realized Residual Variance:

In [ ]:
trueVarE

## The EM Algorithm


In [ ]:
#Construct right hand side of the equations
  XY=t(X)%*%Y
  ZY=t(Z)%*%Y
  RHS=c(XY,ZY)

  #Construct left hand side of the equations
  XX=t(X)%*%X
  XZ=t(X)%*%Z
  ZX=t(Z)%*%X
  ZZ=t(Z)%*%Z
  LHS=matrix(0,(nLocs+nLines),(nLocs+nLines))
  LHS[1:nLocs,1:nLocs]=XX
  LHS[(nLocs+1):(nLocs+nLines),1:nLocs]=ZX
  LHS[1:nLocs,(nLocs+1):(nLocs+nLines)]=XZ

  #initialize with starting value
  Gi=diag(alpha,nLines)
  ZZG=ZZ+Gi

  #start iterations
  iter=1
  delta=100
  while(delta>criteria ){
    print("iteration")
    print(iter)
    iter=iter+1

    #update LHS
    LHS[(nLocs+1):(nLocs+nLines),(nLocs+1):(nLocs+nLines)]=ZZG
    LHSi=solve(LHS)

    #solve for fixed and random effects
    sol=LHSi%*%RHS
    bhat=sol[1:nLocs]
    ahat=sol[(nLocs+1):(nLocs+nLines)]

    #estimate residual variance
    Ve=(t(Y)%*%Y-t(bhat)%*%t(X)%*%Y-t(ahat)%*%t(Z)%*%Y)/(length(Y)-nLocs)

    #estimate genetic variance
    C22=LHSi[(nLocs+1):(nLocs+nLines),(nLocs+1):(nLocs+nLines)]
    Va=(t(ahat)%*%ahat+sum(diag(C22))*Ve)/nLines
    delta=abs(as.double(Ve)/as.double(Va)-alpha)
    #update alpha and G inverse
    alpha=as.double(Ve)/as.double(Va)
    Gi=diag(alpha,nLines)
    ZZG=ZZ+Gi

    print("residual variance estimate")
    print(as.double(Ve))
    print("additive variance estimate")
    print(as.double(Va))

  }

## Examining The Solutions

Realized Genetic Variance:

In [ ]:
trueVarA

Estimated Genetic Variance:

In [ ]:
Va

Realized Residual Variance:

In [ ]:
trueVarE

Estimated Residual Variance:

In [ ]:
Ve

True $\alpha$ :

In [ ]:
trueVarE/trueVarA

Estimated $\alpha$ :

In [ ]:
Ve/Va

Accuracy of BLUPS:

In [ ]:
cor(lines, ahat)